# VISIONX: Vehicle Classifier - Train, Validate, Test

This notebook trains a custom Convolutional Neural Network (CNN) from scratch to classify vehicle types (Bus, Motorcycle, Car, Truck) on the `Vehicles.v1i.multiclass` dataset.

Works seamlessly on **both local environment and Google Colab** (with GPU acceleration).

## 1. Check GPU / Hardware Device

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
else:
    print("Using CPU for execution.")

## 2. Environment & Dataset Setup (Colab & Local Compatible)

Automatically handles dataset path for local execution and Google Colab.

In [ ]:
import os, sys, importlib

# If running in Colab, mount Google Drive dynamically without static import errors
if 'google.colab' in sys.modules or os.path.exists('/content'):
    try:
        colab_drive = importlib.import_module('google.colab.drive')
        colab_drive.mount('/content/drive')
        DATASET_ZIP = '/content/drive/MyDrive/Vehicles.v1i.multiclass.zip'
        if os.path.exists(DATASET_ZIP):
            !unzip -q "$DATASET_ZIP" -d /content/Vehicles.v1i.multiclass
            print("Dataset unzipped from Drive.")
    except Exception as e:
        print("Colab setup note:", e)
else:
    print("Local environment detected. Dataset will be loaded from project directory.")

## 3. Imports and Configuration

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Automatic Dataset Path Resolution
candidate_paths = [
    "/content/Vehicles.v1i.multiclass",
    "Vehicles.v1i.multiclass",
    "../Vehicles.v1i.multiclass",
    os.path.join(os.path.dirname(os.getcwd()), "Vehicles.v1i.multiclass"),
    r"c:\Users\lingeshwaran.k\Downloads\AIML PROJECT\Vehicles.v1i.multiclass"
]

DATASET_PATH = next((p for p in candidate_paths if os.path.exists(p)), "Vehicles.v1i.multiclass")

BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "vehicle_classifier.pth"

torch.manual_seed(42)
print(f"Using device: {DEVICE}")
print(f"Dataset path: {DATASET_PATH}")

## 4. Custom Dataset Definition

In [ ]:
class VehicleDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.annotations.columns = self.annotations.columns.str.strip()

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        img_name = str(self.annotations.iloc[index, 0]).strip()
        img_path = os.path.join(self.root_dir, img_name)

        try:
            image = Image.open(img_path).convert("RGB")
        except FileNotFoundError:
            image = Image.new("RGB", (128, 128))

        labels = self.annotations.iloc[index, 1:].values.astype('float32')
        labels = torch.tensor(labels)

        if self.transform:
            image = self.transform(image)

        return image, labels

## 5. Custom CNN Architecture (No Pretrained Models)

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 64x64

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 32x32

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 16x16
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

## 6. Data Loaders & Augmentations

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def build_loader(split_name, transform, shuffle):
    csv_path = os.path.join(DATASET_PATH, split_name, '_classes.csv')
    img_dir = os.path.join(DATASET_PATH, split_name)
    if not os.path.exists(csv_path):
        print(f"Warning: Split '{split_name}' not found at {csv_path}")
        return None, None
    dataset = VehicleDataset(csv_file=csv_path, root_dir=img_dir, transform=transform)
    loader = DataLoader(dataset=dataset, batch_size=BATCH_SIZE, shuffle=shuffle)
    return dataset, loader

train_dataset, train_loader = build_loader("train", train_transform, shuffle=True)
valid_dataset, valid_loader = build_loader("valid", eval_transform, shuffle=False)
test_dataset, test_loader = build_loader("test", eval_transform, shuffle=False)

num_classes = len(train_dataset.annotations.columns) - 1
class_names = list(train_dataset.annotations.columns)[1:]
print(f"Classes ({num_classes}): {class_names}")
print(f"Train samples: {len(train_dataset)}, Valid: {len(valid_dataset)}, Test: {len(test_dataset)}")

## 7. Model Training Loop

In [ ]:
model = CustomCNN(num_classes=num_classes).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

epoch_losses = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    batch_count = 0
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        batch_count += 1

    avg_loss = running_loss / batch_count
    epoch_losses.append(avg_loss)
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Average Loss: {avg_loss:.4f}")

torch.save(model.state_dict(), MODEL_PATH)
print("\nTraining complete! Model saved to:", MODEL_PATH)

## 8. Training Loss Curve Visualization

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(1, len(epoch_losses) + 1), epoch_losses, 'b-o', linewidth=2, markersize=8)
plt.title("VISIONX - Training Loss Curve", fontsize=14, fontweight='bold')
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss (BCEWithLogitsLoss)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(range(1, len(epoch_losses) + 1))
plt.tight_layout()
plt.savefig("training_loss.png", dpi=150)
plt.show()

## 9. Comprehensive Model Evaluation & Confusion Matrix

In [ ]:
def evaluate_dataset(model, loader, split_name="Validation"):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            all_preds.append(preds)
            all_labels.append(labels.numpy().astype(int))

    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    real_names = [c for c in class_names if c != '0']
    real_indices = [i for i, c in enumerate(class_names) if c != '0']

    print(f"=== {split_name} Classification Report ===")
    print(classification_report(all_labels[:, real_indices], all_preds[:, real_indices], target_names=real_names, zero_division=0))
    return all_preds, all_labels, real_names, real_indices

val_preds, val_labels, real_names, real_indices = evaluate_dataset(model, valid_loader, "Validation")
test_preds, test_labels, _, _ = evaluate_dataset(model, test_loader, "Test")

# Confusion Matrix Plot
fig, axes = plt.subplots(1, len(real_names), figsize=(5 * len(real_names), 4))
fig.suptitle("Per-Class Confusion Matrices (Validation Set)", fontsize=14, fontweight='bold')
for ax, idx, name in zip(axes, real_indices, real_names):
    cm = confusion_matrix(val_labels[:, idx], val_preds[:, idx], labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No", "Yes"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()